In [1]:
import os
import numpy as np
import glob
import os.path as op
import nibabel as nib
from ipyparallel import Client

In [2]:
home_dir =  '/home/dnl/habitization/'
subs = home_dir + 'subjects.txt'
subs = list(np.loadtxt(subs,str))
subs = ['HAB01']

In [3]:
engines = Client()

TimeoutError: Hub connection request timed out

In [ ]:
def prep_fieldmaps(sub):        
    cal_dir = op.join(home_dir,'data',sub ,'cal')

    for sesh in ['a','b']:

        #first drop T1 saturation scans
        cals = glob.glob(cal_dir + '/cal_' + sesh + '*')
        sliced_scans = []
        for pe1 in cals:
            sliced = pe1[:-7] + '_slice'
            sliced_scans.append(sliced)
            cmd = ['fslroi',pe1,sliced,'4','1']
            cmd = ' '.join(cmd)
            os.system(cmd)

        #pe0 from main scans
        for run in range(1,7):
            out_pe0 = cal_dir + '/pe0_slice_' + sesh + str(run)
            in_f =  op.join(home_dir,'data',sub,'func',
                            'run_' + sesh + str(run) + '.nii.gz')
            data = nib.load(in_f)
            idx = data.shape[-1]//2 #middle volume
            cmd = ['fslroi',in_f,out_pe0,str(idx),'1']
            cmd = ' '.join(cmd)
            os.system(cmd) 

            #merge scans
            out_f = cal_dir + '/merged_' + sesh + str(run)
            if run <= 3: #get cal scan from that half of scanning
                idx = 0
            else:
                idx = 1
            cmd = ['fslmerge','-t',out_f,
                   out_pe0,
                   sliced_scans[idx]]
            cmd = ' '.join(cmd)
            os.system(cmd)

            #motion correct
            cmd = ['mcflirt','-in',out_f,'-refvol','0'] #register to functional run
            os.system(' '.join(cmd))

In [ ]:
sub_engines = engines[0:11] #less if you want to use fewer
sub_engines.push(dict(home_dir = home_dir))
sub_engines.execute('import os.path as op')
sub_engines.execute('import numpy as np')
sub_engines.execute('import nibabel as nib')
with sub_engines.sync_imports():
    import os
    import glob
    
output = sub_engines.map_sync(prep_fieldmaps,subs)

In [ ]:
##check cal
for sub in subs:
    cal_dir = op.join(home_dir,'data',sub ,'cal')
    for sesh in ['a','b']:
        for run in range(1,7):
            out_f = cal_dir + '/merged_' + sesh + str(run) + '_mcf.nii.gz'
            
            if not os.path.exists(out_f):
                print 'no file',sub,sesh,run

            elif nib.load(out_f).shape[-1] != 2:
                print 'wrong size',sub,sesh,run

    

In [ ]:
##make sym links for lyman analyis
new_name = {'a1':'01',
           'a2':'02',
            'a3':'03',
           'a4':'04',
            'a5':'05',
           'a6':'06',
            'b1':'07',
           'b2':'08',
            'b3':'09',
           'b4':'10',
            'b5':'11',
           'b6':'12'}
            
for sub in subs:
    sub_dir = home_dir + '/data/'+ sub 
    
    new_func = home_dir + '/data/'+ sub + '/func/lyman/'
    make_new_dir(new_func)
  
    new_cal= home_dir + '/data/'+ sub + '/cal/lyman/'
    make_new_dir(new_cal)
    
    for sesh in ['a','b']:
        for run in map(str,range(1,7)):
            
            func = sub_dir + '/func/run_' + sesh + run + '.nii.gz'
            new_f = new_func + 'run_' + new_name[sesh + run] + '.nii.gz'
            cmd = ['ln','-s',func,new_f]
            cmd = ' '.join(cmd)
            os.system(cmd)

            cal = sub_dir + '/cal/merged_' + sesh + str(run) + '_mcf.nii.gz'
            new_f = new_cal + 'run_' + new_name[sesh + run] + '.nii.gz'
            cmd = ['ln','-s',cal,new_f]
            cmd = ' '.join(cmd)
            os.system(cmd)